# Day 12 — Multi-Agent Venture Intelligence System (Layer 12)

The platform stops being a single reasoner and becomes an **investment committee**: six specialist
agents each analyze a campaign from one lens, an orchestrator runs them on **one shared
intelligence**, and a consensus engine resolves their stances into a **committee decision** — with a
risk veto and surfaced conflicts.

**Honesty (C-7).** The six agents are **not independent experts** — they all read the *same* Day-11
`VentureIntelligence`, so their stances are **correlated views of one signal set**, and consensus
says so (confidence is **not** inflated by agent count; the CompetitionAgent is down-weighted for its
overlap with the FinancialAgent). Deterministic, **no LLM, no faiss**. Founder background,
financials, market size, moat, and regulatory risk are **deferred**, never fabricated. Logic in
`src/agents/`.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import pandas as pd, numpy as np

# The consensus engine builds the orchestrator (6 agents) on ONE warm faiss-free stack.
from src.agents.consensus import ConsensusEngine
ce = ConsensusEngine()

df = pd.read_csv("data/processed/kickstarter_processed.csv"); ti = np.load("artifacts/splits/test_index.npy")
row = df.iloc[ti][df.iloc[ti]["usd_goal_real"] > 12000].sample(1, random_state=4).iloc[0].to_dict()
D = ce.run(row)                              # orchestrate 6 agents -> committee decision

print("CAMPAIGN: %s | %s | goal $%.0f" % (row["name"], row["main_category"], row["usd_goal_real"]))
print("COMMITTEE: %s (score %+.2f, confidence %.0f%%, agreement %.0f%%)" %
      (D["committee_recommendation"], D["committee_score"],
       D["overall_confidence"]*100, D["agreement_level"]*100))

CAMPAIGN: Artificer - Board Game | Games | goal $30516
COMMITTEE: Do Not Invest (score -0.69, confidence 78%, agreement 83%)


## The six specialist agents (the committee table)

Each agent scores the campaign through one lens and emits a uniform report — a 0-100 score (higher =
more favorable), a stance, its own confidence, and grounded findings. Note the **reframes**: Market =
category (not TAM), Founder = creator execution (not the person), Financial = funding viability (not
P&L), Competition = cohort (not moat).

In [2]:
print("%-16s  %-9s %5s  %-6s  %s" % ("AGENT", "STANCE", "SCORE", "CONF", "TOP FINDING"))
print("-" * 100)
for r in D["agent_reports"]:
    print("%-16s  %-9s %5d  %.2f    %s" %
          (r["agent"], r["stance"], r["score"], r["confidence"], r["findings"][0][:54]))

AGENT             STANCE    SCORE  CONF    TOP FINDING
----------------------------------------------------------------------------------------------------
MarketAgent       positive     72  0.87    Games campaigns fund at 44% vs a 40% platform baseline
FounderAgent      negative     36  0.83    The creator left ~42 points of controllable upside unr
FinancialAgent    negative     24  0.78    Goal $30,516 is 6.1x the category winners' median ($5,
CompetitionAgent  negative     17  0.99    Ask for less: funded campaigns' median goal is $5,000 
RiskAgent         negative     27  0.78    Overall risk: High. Modeled failure risk 76%.
InvestmentAgent   negative     35  0.82    1 positive / 0 cautious / 4 negative across 5 speciali


## Phase H — Orchestration (one warm stack, one aggregation)

All six agents run on a **single** aggregated intelligence and a single loaded model — the
investment agent is fed the five specialists' reports rather than re-running them.

In [3]:
ids = {id(a.agg.cf) for a in ce.orchestrator.specialists} | {id(ce.orchestrator.investment.agg.cf)}
print("all agents share ONE model in memory:", len(ids) == 1)
print("no faiss / torch imported        :", all(m not in sys.modules for m in ("faiss", "torch")))
print("agents run                       :", [r["agent"].replace("Agent", "") for r in D["agent_reports"]])

all agents share ONE model in memory: True
no faiss / torch imported        : True
agents run                       : ['Market', 'Founder', 'Financial', 'Competition', 'Risk', 'Investment']


## Phase I — Consensus & the committee decision

A **confidence-weighted stance vote** (positive +1 / cautious 0 / negative -1), distinct from the
investment agent's score-mean. It surfaces dissenters and applies a **risk veto**.

In [4]:
print("RECOMMENDATION :", D["committee_recommendation"], "(score %+.2f)" % D["committee_score"])
print("confidence     : %.0f%%   agreement: %.0f%%" % (D["overall_confidence"]*100, D["agreement_level"]*100))
print("\nvotes:")
for v in D["agent_votes"]:
    print("   %-16s %-9s conf %.2f  weight %.1f" % (v["agent"], v["stance"], v["confidence"], v["weight"]))
print("\nconflicts (dissent from majority):", [c["agent"] for c in D["conflicts"]] or "none")
print("risk veto:", D["risk_veto"])
print("\nrationale:", D["rationale"])

RECOMMENDATION : Do Not Invest (score -0.69)
confidence     : 78%   agreement: 83%

votes:
   MarketAgent      positive  conf 0.87  weight 1.0
   FounderAgent     negative  conf 0.83  weight 1.0
   FinancialAgent   negative  conf 0.78  weight 1.0
   CompetitionAgent negative  conf 0.99  weight 0.8
   RiskAgent        negative  conf 0.78  weight 1.5
   InvestmentAgent  negative  conf 0.82  weight 1.5

conflicts (dissent from majority): ['MarketAgent']
risk veto: {'eligible': True, 'applied': False}

rationale: Committee recommendation: Do Not Invest (score -0.69; 1 positive / 5 negative of 6 agents). Dissent: MarketAgent (positive).


## The risk veto in action

Naturally, a decisive risk stance usually drags the whole committee down. To *show* the veto, here's
a synthetic committee — five optimists and one decisive-negative risk agent. The weighted tally is
positive, but the veto caps it: **you don't invest past a red flag.**

In [5]:
synthetic = [
    {"agent": "MarketAgent",      "stance": "positive", "confidence": 0.90, "score": 75, "deferred": {}},
    {"agent": "FounderAgent",     "stance": "positive", "confidence": 0.80, "score": 70, "deferred": {}},
    {"agent": "FinancialAgent",   "stance": "positive", "confidence": 0.85, "score": 72, "deferred": {}},
    {"agent": "CompetitionAgent", "stance": "positive", "confidence": 0.80, "score": 68, "deferred": {}},
    {"agent": "RiskAgent",        "stance": "negative", "confidence": 0.95, "score": 20, "deferred": {}},
    {"agent": "InvestmentAgent",  "stance": "positive", "confidence": 0.85, "score": 71, "deferred": {}},
]
sd = ce.decide(synthetic)
print("weighted tally score: %+.2f  (positive -> would be Strong Invest / Proceed)" % sd["committee_score"])
print("committee decision  :", sd["committee_recommendation"], "  <- risk veto:", sd["risk_veto"]["applied"])
print(sd["rationale"])

weighted tally score: +0.52  (positive -> would be Strong Invest / Proceed)
committee decision  : Monitor Progress   <- risk veto: True
Committee recommendation: Monitor Progress (score +0.52; 5 positive / 1 negative of 6 agents). Risk veto applied — capped at Monitor Progress despite a positive tally. Dissent: RiskAgent (negative).


## The honesty boundary

The committee is a structured aggregation of **correlated** perspectives, and it records exactly what
it refuses to assess.

In [6]:
print("HONESTY NOTE:\n ", D["honesty_note"])
print("\nCompetitionAgent weight (down-weighted for overlap with FinancialAgent):",
      next(v["weight"] for v in D["agent_votes"] if v["agent"] == "CompetitionAgent"))
print("\nDEFERRED across the committee (%d items):" % len(D["deferred"]))
for k in list(D["deferred"])[:10]:
    print("   -", k)
print("   ... and %d more" % (len(D["deferred"]) - 10))

HONESTY NOTE:
  Committee = confidence-weighted stance vote over correlated specialist views of ONE shared signal set (not independent experts) — so confidence is NOT inflated by agent count, and the CompetitionAgent is down-weighted for overlap with the FinancialAgent. A decisive RiskAgent negative vetoes a positive committee. Model-simulated, not causal.

CompetitionAgent weight (down-weighted for overlap with FinancialAgent): 0.8

DEFERRED across the committee (35 items):
   - market_size_tam
   - industry_cagr
   - addressable_market
   - industry_growth_rate
   - founder_experience
   - team_composition
   - leadership
   - advisory_board
   - domain_expertise
   - founder_track_record
   ... and 25 more


## Phase J — Serialization (content-hash identity + verify)

The committee is a **policy** over the Day-4 model + Day-8/11 engines — persisted as JSON, not a
pickle. Its dependency hashes form a Merkle-style chain over the whole Day-4→11 stack.

In [7]:
from src.agents.serialization import AgentSystemSerializer
ser = AgentSystemSerializer().build(); ser.save()
v = ser.verify()
print(ser.describe())
print("verify all_ok:", v["all_ok"], "(%d/%d checks pass)" % (sum(v["checks"].values()), len(v["checks"])))

Multi-agent system v1.0.0 (content f235c7797f46…): layer 12; 6 agents, 3 deps hashed, 8 artifacts, 22 items deferred.
verify all_ok: True (12/12 checks pass)


## Day 12 complete — Layer 12 ✅

The platform now emulates an **investment committee**: six specialist agents on one shared
intelligence, a confidence-weighted consensus with a risk veto and surfaced conflicts, content-hash
verified. The agents are **correlated views of one signal set, not independent experts** — and the
committee says so rather than inflating confidence. Founder background, financials, market size,
moat, and regulatory risk remain **deferred, not invented**. Model-simulated, not causal.

Foundation for **Layer 13 — Professional Report Generation** (Founder & Investor reports / memo).